<a href="https://colab.research.google.com/github/Asaf21S/constrained-flow-matching/blob/main/constrained-flow-matching/constraints_distillation/notebooks/constrained_fm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imports

In [ ]:
!pip install torchcfm

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, TensorDataset
from torchcfm.conditional_flow_matching import ConditionalFlowMatcher, ExactOptimalTransportConditionalFlowMatcher
import matplotlib.pyplot as plt
import numpy as np
from tqdm.auto import tqdm
import math

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

# Dataset

In [ ]:
RNG = np.random.default_rng()

In [ ]:
def get_1d_gaussian_data(peaks: list[tuple[float, float]], n_samples: int, rng=RNG):
    means = np.array([p[0] for p in peaks])
    stds = np.array([p[1] for p in peaks])
    weights = np.full(len(peaks), 1.0 / len(peaks))

    # Pick which peak each sample belongs to
    choices = rng.choice(len(peaks), p=weights, size=n_samples)

    # Generate standard normal samples (mean=0, std=1)
    base_samples = rng.standard_normal(size=n_samples)

    # Scale and shift the base samples
    mixture_samples = (base_samples * stds[choices]) + means[choices]

    return torch.from_numpy(mixture_samples).float()

In [ ]:
peaks = [(2, 1), (-2, 2)]
peaks_str = "".join([f"_{mean}_{std}" for mean, std in peaks])

In [ ]:
real_data = get_1d_gaussian_data(peaks=peaks, n_samples=100000)

In [ ]:
# plot data:
plt.hist(real_data, bins=100)
plt.show()

# General Functions

## visualization

In [ ]:
def visualize_fm(sampled_data, title="1D Flow Matching Results", alpha=0.5, true_data=None, boundaries=None, center_symmetry=None):
    """
    Visualizes Flow Matching trajectories and final distributions for 1D data.
    Dynamically creates a 3rd subplot for zoomed-in boundary analysis if needed.
    """
    if sampled_data.ndim == 3 and sampled_data.shape[-1] == 1:
        sampled_data = sampled_data.squeeze(-1)

    start_points = sampled_data[0]
    end_points = sampled_data[-1]
    n_steps, n_samples = sampled_data.shape
    n_plot = min(100, n_samples)

    # Dynamically choose 2 or 3 subplots
    num_plots = 3 if boundaries is not None else 2
    fig, axes = plt.subplots(1, num_plots, figsize=(6 * num_plots, 5))

    if num_plots == 2:
        ax_traj, ax_macro = axes
        ax_micro = None
    else:
        ax_traj, ax_macro, ax_micro = axes

    # ==========================================
    # 1. Flow Plot: Time (t) vs. Position (x)
    # ==========================================
    ax_traj.set_title("1D Trajectories Over Time")
    t_values = np.linspace(0, 1, n_steps)
    ax_traj.plot(t_values, sampled_data[:, :n_plot], color='black', alpha=0.15)
    ax_traj.scatter(np.zeros(n_plot), start_points[:n_plot], c='red', s=15, label='Start (Noise)', zorder=3)
    ax_traj.scatter(np.ones(n_plot), end_points[:n_plot], c='blue', s=15, label='End (Data)', zorder=3)

    if boundaries is not None:
        min_val, max_val = boundaries
        ax_traj.axhline(y=min_val, color='green', linestyle='--', linewidth=2, zorder=4, label='Boundaries')
        ax_traj.axhline(y=max_val, color='green', linestyle='--', linewidth=2, zorder=4)

    if center_symmetry is not None:
        ax_traj.axhline(y=center_symmetry, color='orange', linestyle='-.', linewidth=2, zorder=4, label='Symmetry Center')

    ax_traj.set_xlabel("Time (t)")
    ax_traj.set_ylabel("Position (x)")
    ax_traj.legend()
    ax_traj.grid(True, alpha=0.3)

    # ==========================================
    # 2. Macro Distribution (Global Context)
    # ==========================================
    # Calculate metrics for the title
    macro_title = title
    if boundaries is not None:
        inside_count = ((end_points >= min_val) & (end_points <= max_val)).sum()
        macro_title += f"\nBoundaries: {inside_count / n_samples * 100:.2f}%"

    if center_symmetry is not None:
        sorted_ends = np.sort(end_points)
        half_N = n_samples // 2
        sym_mse = np.mean((sorted_ends[:half_N] + sorted_ends[-half_N:][::-1] - 2 * center_symmetry) ** 2)
        macro_title += f"\nSymmetry MSE: {sym_mse:.5f}"

    ax_macro.set_title(macro_title)

    if true_data is not None:
        if true_data.ndim == 2 and true_data.shape[-1] == 1:
            true_data = true_data.squeeze(-1)
        ax_macro.hist(true_data, bins=100, density=True, color='magenta', alpha=0.2, label="Original Prior")

    ax_macro.hist(end_points, bins=100, density=True, color='blue', alpha=alpha, label="Generated Data")

    if boundaries is not None:
        ax_macro.axvline(x=min_val, color='green', linestyle='--', linewidth=2, zorder=4)
        ax_macro.axvline(x=max_val, color='green', linestyle='--', linewidth=2, zorder=4)

    if center_symmetry is not None:
        ax_macro.axvline(x=center_symmetry, color='orange', linestyle='-.', linewidth=2, zorder=4)

    ax_macro.set_xlabel("Position (x)")
    ax_macro.set_ylabel("Density")
    ax_macro.legend()
    ax_macro.grid(True, alpha=0.3)

    # ==========================================
    # 3. Micro Distribution (Zoomed Boundary Fit)
    # ==========================================
    if ax_micro is not None:
        ax_micro.set_title("Zoom: Conditional Fit (Shape Matching)")

        # Only plot points that actually made it inside the boundaries
        generated_inside = end_points[(end_points >= min_val) & (end_points <= max_val)]

        if true_data is not None:
            ideal_dist = true_data[(true_data >= min_val) & (true_data <= max_val)]
            # Use a step outline for the ideal distribution so it doesn't muddy the blue fill
            ax_micro.hist(ideal_dist, bins=100, density=True, histtype='step',
                          color='black', linewidth=2, linestyle='-', zorder=5, label="Ideal Shape (Target)")

        ax_micro.hist(generated_inside, bins=100, density=True, color='blue', alpha=alpha, label="Generated (Inside)")

        ax_micro.axvline(x=min_val, color='green', linestyle='--', linewidth=2, zorder=4)
        ax_micro.axvline(x=max_val, color='green', linestyle='--', linewidth=2, zorder=4)

        # Set the tight x-limits
        margin = (max_val - min_val) * 0.1
        ax_micro.set_xlim(min_val - margin, max_val + margin)

        ax_micro.set_xlabel("Position (x)")
        ax_micro.legend()
        ax_micro.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## Feature Importance

In [ ]:
def calculate_permutation_importance(model, dataloader, criterion, feature_names):
    """
    Calculates permutation feature importance for a model with an arbitrary number of inputs.

    Args:
        model: The trained PyTorch model.
        dataloader: DataLoader containing the validation/test data.
        criterion: The loss function.
        feature_names: List of strings representing the names of the features in order.
    """
    model.eval()

    # Calculate Baseline Loss
    baseline_loss = 0.0
    with torch.no_grad():
        for batch in dataloader:
            # Move entire batch to device
            batch = [b.to(device) for b in batch]

            # The last element is the target, everything else is a feature
            features = batch[:-1]
            target = batch[-1]

            # Unpack the features list directly into the model arguments
            pred = model(*features)

            # Calculate loss (Ensure target shape matches pred shape!)
            baseline_loss += criterion(pred, target.view_as(pred)).item()

    baseline_loss /= len(dataloader)

    print(f"Baseline MSE Loss: {baseline_loss:.6f}\n")

    # Permute each feature and measure the drop in performance
    importances = {}

    for i, feature_name in enumerate(feature_names):
        permuted_loss = 0.0

        with torch.no_grad():
            for batch in dataloader:
                batch = [b.to(device) for b in batch]
                features = batch[:-1]
                target = batch[-1]

                # Shuffle the chosen feature along the batch dimension
                indices = torch.randperm(features[i].size(0), device=device)
                features[i] = features[i][indices]

                # Pass corrupted features to model using unpacking
                pred = model(*features)

                permuted_loss += criterion(pred, target.view_as(pred)).item()

        permuted_loss /= len(dataloader)

        # Importance = How much worse the loss got
        increase_in_loss = permuted_loss - baseline_loss
        importances[feature_name] = increase_in_loss

        print(f"Shuffled {feature_name:<10} | New Loss: {permuted_loss:.6f} | Error Increase: +{increase_in_loss:.6f}")

    return importances

In [ ]:
def plot_feature_importance(importances, save_path="student_feature_importance.png"):
    """
    Creates a professional horizontal bar chart for feature importances.

    Args:
        importances (dict): The dictionary returned by calculate_permutation_importance.
        save_path (str): Where to save the generated image for your README.
    """
    sorted_importances = sorted(importances.items(), key=lambda item: item[1])
    features = [item[0] for item in sorted_importances]
    values = [item[1] for item in sorted_importances]

    plt.figure(figsize=(10, 6))

    bars = plt.barh(features, values, color='#5A9BD5', edgecolor='black', alpha=0.8)

    plt.xlabel("Increase in MSE Loss (Importance)", fontsize=12, labelpad=10)
    plt.title("Permutation Feature Importance", fontsize=14, fontweight='bold', pad=15)

    plt.grid(axis='x', linestyle='--', alpha=0.5, zorder=0)

    for bar in bars:
        width = bar.get_width()
        padding = max(values) * 0.01
        plt.text(
            width + padding,
            bar.get_y() + bar.get_height() / 2,
            f"+{width:.4f}",
            va='center',
            ha='left',
            fontsize=10,
            color='black'
        )

    plt.xlim(0, max(values) * 1.15)

    plt.tight_layout()

    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"Plot saved to {save_path}")

    plt.show()

# Conditional Boundary FM

In [ ]:
width_range=(0.1, 5.0)

In [ ]:
class ConditionalBoundaryFlowMatcher(nn.Module):
    def __init__(self):
        super().__init__()
        # Input dim is 4: [x_t, t, dist_left, dist_right]
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.SiLU(),
            nn.Linear(64, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 64),
            nn.SiLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_t, t, a, b):
        d_left = x_t - a
        d_right = b - x_t

        x_t = x_t.view(-1, 1)
        t = t.view(-1, 1)
        d_left = d_left.view(-1, 1)
        d_right = d_right.view(-1, 1)

        inputs = torch.cat([x_t, t, d_left, d_right], dim=-1)
        return self.net(inputs)

In [ ]:
def train_conditional_fm(dataloader, epochs=25, width_range=(0.1, 5.0)):
    model = ConditionalBoundaryFlowMatcher().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    loss_list = []

    print("Starting Conditional Flow Matching Training...")

    for epoch in tqdm(range(epochs)):
        model.train()
        total_loss = 0.0

        for batch in dataloader:
            x_1 = batch[0].to(device).view(-1, 1)  # real data
            batch_size = x_1.shape[0]

            x_0 = torch.randn_like(x_1)  # noise
            t = torch.rand((batch_size, 1), device=device)

            # Boundaries' width
            d = torch.empty((batch_size, 1), device=device).uniform_(*width_range)

            # Offset from min' boundary
            o = torch.rand((batch_size, 1), device=device) * d

            a = x_1 - o
            b = a + d

            x_t = (1 - t) * x_0 + t * x_1
            target_v = x_1 - x_0

            optimizer.zero_grad()
            pred_v = model(x_t, t, a, b)

            loss = criterion(pred_v, target_v)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(dataloader)
        loss_list.append(avg_loss)

        if epoch % 5 == 0 or epoch == epochs - 1:
            print(f"Epoch {epoch}/{epochs} | Loss: {avg_loss:.6f}")

    print("Training Complete!")
    return model, loss_list

In [ ]:
batch_size = 2048
real_data_2d = real_data.view(-1, 1)
train_dataset = TensorDataset(real_data_2d)
train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True
)

print(f"Total batches per epoch: {len(train_loader)}")
for batch in train_loader:
    x_1_batch = batch[0]
    print(f"Batch shape: {x_1_batch.shape}")
    break

In [ ]:
conditional_fm_model, loss_list_fm = train_conditional_fm(train_loader, epochs=100, width_range=width_range)

In [ ]:
plt.plot(loss_list_fm)
plt.show()

In [ ]:
def sample_trajectory(model, boundaries, n_samples, steps=100):
    x = torch.randn(n_samples, 1, device=device)
    dt = 1.0 / steps
    a, b = boundaries

    traj = [x.cpu()]
    model.eval()
    with torch.no_grad():
        for i in range(steps):
            t = i * dt
            t_batch = torch.full((n_samples, 1), t, device=device)
            v = model(x, t_batch, a, b)
            x = x + v * dt
            traj.append(x.cpu())

    return torch.stack(traj).numpy()  # Final shape: [steps + 1, n_samples, 2]

In [ ]:
boundaries = (-2, 2)
sampled_data = sample_trajectory(conditional_fm_model, boundaries=boundaries, n_samples=100000)

In [ ]:
visualize_fm(
    sampled_data=sampled_data,
    true_data=real_data[:100000],
    title="Constrained FM vs Real Data",
    boundaries=boundaries
)

In [ ]:
boundaries = (-4, 0)
sampled_data = sample_trajectory(conditional_fm_model, boundaries=boundaries, n_samples=100000)

In [ ]:
visualize_fm(
    sampled_data=sampled_data,
    true_data=real_data[:100000],
    title="Constrained FM vs Real Data",
    boundaries=boundaries
)

In [ ]:
boundaries = (0, 2.5)
sampled_data = sample_trajectory(conditional_fm_model, boundaries=boundaries, n_samples=100000)

In [ ]:
visualize_fm(
    sampled_data=sampled_data,
    true_data=real_data[:100000],
    title="Constrained FM vs Real Data",
    boundaries=boundaries
)

In [ ]:
boundaries = (-1, 1)
sampled_data = sample_trajectory(conditional_fm_model, boundaries=boundaries, n_samples=100000)

In [ ]:
visualize_fm(
    sampled_data=sampled_data,
    true_data=real_data[:100000],
    title="Constrained FM vs Real Data",
    boundaries=boundaries
)